Resultaten wegschrijven in genormaliseerd datamodel. Voorlopig gesimuleerd als sqlite. Connectie met LSVI databank of andere masterdata nog uit te klaren?

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sqlite3
from datetime import datetime
from pathlib import Path
import pandas as pd

from arcgis.gis import GIS
import sys

# 1. Bepaal het absolute pad van de bovenliggende map (de project root)
project_root = str(Path.cwd().parent)

# 2. Voeg de project root toe aan sys.path, zodat Python de 'src' map kan vinden
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import src.utils as utils
import json


### Connectie naar databank

In [ ]:
# Aanmaken 
def init_database(db_path="lsvi_resultaten_slank.sqlite"):
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    cursor.execute("PRAGMA foreign_keys = ON;")
    
    # Tabel 1: WaarnemingEvent
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS waarneming_event (
        parent_id TEXT,
        collectie_id TEXT PRIMARY KEY,
        global_id TEXT,
        tijdstip DATETIME,
        waarnemer TEXT,
        locatie_x REAL,
        locatie_y REAL,
        EPSG INTEGER,
        doel_habitattype TEXT,
        bwk_id TEXT,
        survey_versie TEXT,
        created_date DATETIME,
        last_edited_date DATETIME
    );
    """)
    
    # Tabel 2: Resultaat (Nu ook voor de losse soorten!)
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS resultaat (
        resultaat_id INTEGER PRIMARY KEY AUTOINCREMENT,
        parent_id TEXT
        collectie_id TEXT,
        voorwaarde_id INTEGER,
        vraag_id TEXT,
        vraag_label TEXT,
        waarde_tekst TEXT,
        waarde_numeriek REAL,
        taxongroep_id INTEGER,
        FOREIGN KEY (parent_id) REFERENCES waarneming_event(parent_id) ON DELETE CASCADE
    );
    """)
    conn.commit()
    return conn

# Helperfunctie om ArcGIS Epoch Milliseconds om te zetten naar een ISO string
def format_agol_date(epoch_ms):
    if epoch_ms is None:
        return None
    # AGOL timestamps zijn in milliseconden, Python verwacht seconden
    return datetime.fromtimestamp(epoch_ms / 1000.0).strftime('%Y-%m-%d %H:%M:%S')

#     conn.commit()
#     conn.close()
#     print("ETL afgerond. De SQLite database is volledig up-to-date.")

### Run ETL

In [ ]:
# Define your local specific paths 
LOCAL_DB = "G:\Mijn Drive\keepass_db.kdbx"
ENTRY_TITLE = "AGOL"

# Run the function
AGOL_USER, AGOL_PASS = utils.load_keepass_credentials(db_path=LOCAL_DB, entry_name=ENTRY_TITLE)


In [ ]:
# Uitvoering
# Joost to do: credentials uit key vault of environment
agol_url = "https://gisservices.inbo.be/portal"
feature_layer_item_id = "3f6c53bf454f42e59f036423f4b1aae8" # Adjust
db_path = "../output/lsvi_resultaten.sqlite"

# Start de volledige pipeline
# run_etl(agol_url, username, password, feature_layer_item_id, db_path)

In [ ]:
map_data_str

In [ ]:
print(f"Verbinden met {agol_url}...")
gis = GIS(agol_url, AGOL_USER, AGOL_PASSWORD)
# Haal de Feature Layer op en lees de data volledig uit
print(f"Feature layer {feature_layer_item_id} downloaden...")
layer_item = gis.content.get(feature_layer_item_id)
feature_layer = layer_item.layers[0]  

# 1
# Load data from repeat tables and index in memory
print("Gerelateerde habitat-tabellen (repeats) downloaden en indexeren...")
related_data_by_parent = {} # Key: parentglobalid (schoon), Value: dict van gecombineerde antwoorden

for table in layer_item.tables:
    table_name = table.properties.get('name', 'Onbekende tabel')
    print(f" -> Bezig met inladen van tabel: {table_name}...")
    
    # Vraag alle records op uit deze specifieke repeat-tabel
    table_result = table.query(where="1=1")
    
    for child_feature in table_result.features:
        c_attrs = child_feature.attributes if child_feature.attributes else {}
        pgid = c_attrs.get('parentrowid')
        
        if pgid:
            # Saneer de GUID (verwijder accolades en zet naar kleine letters voor een harde match)
            pgid_clean = str(pgid).lower().strip('{}')
            
            if pgid_clean not in related_data_by_parent:
                related_data_by_parent[pgid_clean] = {}
            
            # Voeg de antwoorden toe aan de bestaande pool voor deze ouder-feature
            related_data_by_parent[pgid_clean].update(c_attrs)

# 2
# Load main feature layer
# Vraag alle records op (1=1) inclusief WGS84 geometrie (out_sr=4326)
features_result = feature_layer.query(where="1=1", out_sr=4326)
print(f"Succesvol {len(features_result.features)} features gedownload. Start verwerking...")

conn = init_database(db_path)
cursor = conn.cursor()

# Loop door ELKE feature uit de layer
for feature in features_result.features:
    geom = feature.geometry if feature.geometry else {}
    attrs = feature.attributes if feature.attributes else {}

    collectie_id = attrs.get('collectie_id')        
    global_id = attrs.get('globalid')
    parent_id = attrs.get('uniquerowid')
    if collectie_id and parent_id:
        print(f"Waarneming {collectie_id} heeft een parent_id: {parent_id}")
        parent_id_clean = str(parent_id).lower().strip('{}')
    else:
        continue

    waarnemer = attrs.get('last_edited_user')
    habitat_keuze = attrs.get('habitat_keuze')
    bwk_id = attrs.get('bwk_id')
    
    created_date_txt = format_agol_date(attrs.get('created_date'))
    last_edited_date_txt = format_agol_date(attrs.get('last_edited_date'))
    
    datum_txt = format_agol_date(attrs.get('datum'))
    uur_txt = attrs.get('uur')  
    tijdstip_waarneming = f"{datum_txt.split(' ')[0]} {uur_txt}" if datum_txt and uur_txt else datum_txt

    x = geom.get('x')
    y = geom.get('y')
    epsg = geom.get('spatialReference', {}).get('wkid')

    # Schrijf het waarneming_event weg
    cursor.execute("""
        INSERT INTO waarneming_event (
            parent_id, collectie_id, global_id,tijdstip, waarnemer, locatie_X, locatie_Y, epsg, doel_habitattype, bwk_id, created_date, last_edited_date
        ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        ON CONFLICT(parent_id) DO UPDATE SET
            collectie_id=excluded.collectie_id,
            global_id=excluded.global_id,
            tijdstip=excluded.tijdstip,
            waarnemer=excluded.waarnemer,
            locatie_X=excluded.locatie_X,
            locatie_Y=excluded.locatie_Y,
            epsg=excluded.epsg,
            doel_habitattype=excluded.doel_habitattype,
            bwk_id=excluded.bwk_id,
            created_date=excluded.created_date,
            last_edited_date=excluded.last_edited_date;
        """, (parent_id, collectie_id, global_id, tijdstip_waarneming, waarnemer, x, y, epsg, habitat_keuze, bwk_id, created_date_txt, last_edited_date_txt))
    
    # Wis oude resultaten voor overschrijven
    cursor.execute("DELETE FROM resultaat WHERE collectie_id = ?", (collectie_id,))

    # 3
    # Merge linked table data into main feature attributes
    # Copy attributes from main feature layer and attach repeat-attributes
    combined_attrs = attrs.copy()

    if parent_id_clean in related_data_by_parent:
        combined_attrs.update(related_data_by_parent[parent_id_clean])

    # Loop over all attributes and add data to table
    for key, value in combined_attrs.items():
            
            
        # Voeg 'parentglobalid' toe aan de systeem-skips
        if key.lower() in ['objectid', 
                           'globalid', 
                           'parentglobalid', 
                           'collectie_id', 
                           'uniquerowid',
                           'datum', 
                           'uur', 
                           'waarnemer', 
                           'habitat_keuze', 
                           'bwk_id', 
                           'created_date', 
                           'created_user', 
                           'last_edited_date', 
                           'last_edited_user',
                           'hab1', 'hab2', 'hab3', 'hab4', 'hab5', 
                           'phab1', 'phab2', 'phab3', 'phab4', 'phab5',
                           'lsvi_opstellen',]:
            continue
        if key.endswith('_note') or key.startswith('note_') or key.startswith('mat_grp_'):
            continue
            
        # FIX: Robuuste Regex-parser voor het extraheren van het VoorwaardeID
        # Zoekt naar het eerste getal direct na 'vraag_' of 'v' (bijv. vraag_1337_... -> 1337, v317_... -> 317)
        voorwaarde_id = None
        match_id = re.search(r'(?:vrg_|v)(\d+)', key)
        if match_id:
            voorwaarde_id = int(match_id.group(1))
        
        # Select_multiple afhandeling (komma-gescheiden soorten)
        if isinstance(value, str) and "," in value:
            soorten = [s.strip() for s in value.split(",")]
            for soort in soorten:
                if soort:
                    cursor.execute("""
                    INSERT INTO resultaat (parent_id, voorwaarde_id, vraag_id, waarde_tekst)
                    VALUES (?, ?, ?, ?);
                    """, (parent_id_clean, voorwaarde_id, key, soort))
        else:
            # Normale single-choice of getal
            waarde_numeriek = None
            try:
                waarde_numeriek = float(value)
            except ValueError:
                pass  
                
            cursor.execute("""
            INSERT INTO resultaat (parent_id, voorwaarde_id, vraag_id, waarde_numeriek)
            VALUES (?, ?, ?, ?);
            """, (parent_id_clean, voorwaarde_id, key, waarde_numeriek))

In [ ]:
combined_attrs

In [ ]:
# related_data_by_parent
# attrs
parent_id_clean = '2bacfdd9-b21e-4186-99db-ab186bcb4265'

combined_attrs = attrs.copy()

if parent_id_clean in related_data_by_parent:
    combined_attrs.update(related_data_by_parent[parent_id_clean])

### Load data export

In [ ]:
sqlite_path = "../output/lsvi_resultaten.sqlite"
conn = sqlite3.connect(sqlite_path)

tables = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;",
    conn
)
print("Tables in database:")
display(tables)

print("\nWaarnemingEvent sample:")
display(pd.read_sql_query("SELECT * FROM waarneming_event LIMIT 20;", conn))

print("\nResultaat sample:")
results = pd.read_sql_query("SELECT * FROM resultaat WHERE collectie_id = '2e5acf0e-cef5-4d29-ad06-96900a597a7b';", conn)
display(results)

conn.close()